# Example: Time-Varying Frequency Response Maps with `sid.freq_map`

This example demonstrates `sid.freq_map`, which estimates how the frequency
response evolves over time by applying spectral analysis to overlapping
segments. Useful for detecting time-varying dynamics (e.g., drifting
poles, changing gains).

In [ ]:
import numpy as np
from scipy.signal import lfilter
import matplotlib.pyplot as plt
import sid

%matplotlib inline

## LTI Baseline: Constant System

For a time-invariant system, the frequency map should be constant along
the time axis.

In [ ]:
rng = np.random.default_rng(10)
N = 4000
u = rng.standard_normal(N)
y = lfilter([1], [1, -0.9], u) + 0.1 * rng.standard_normal(N)

result_lti = sid.freq_map(y, u, segment_length=512)

fig, ax = plt.subplots()
sid.map_plot(result_lti, plot_type='magnitude', ax=ax)
ax.set_title('LTI System: Magnitude Should Be Constant Along Time')
plt.tight_layout()
plt.show()

## Time-Varying System: Drifting Pole

Simulate a first-order system `x(k+1) = a(k)*x(k) + u(k)` where the
pole `a(k)` ramps from 0.5 to 0.95 over time. This creates a system that
becomes more resonant (higher gain at low frequencies) as time progresses.

In [ ]:
rng = np.random.default_rng(20)
N = 6000
u_tv = rng.standard_normal(N)
a_k = np.linspace(0.5, 0.95, N)  # time-varying pole

y_tv = np.zeros(N)
for k in range(1, N):
    y_tv[k] = a_k[k] * y_tv[k - 1] + u_tv[k]
y_tv += 0.05 * rng.standard_normal(N)  # small measurement noise

result_tv = sid.freq_map(y_tv, u_tv, segment_length=512, overlap=384)

fig, ax = plt.subplots()
sid.map_plot(result_tv, plot_type='magnitude', ax=ax)
ax.set_title('Time-Varying System: Pole Drifts from 0.5 to 0.95')
plt.tight_layout()
plt.show()

## Coherence Map

Coherence shows how the signal-to-noise ratio evolves over time.

In [ ]:
fig, ax = plt.subplots()
sid.map_plot(result_tv, plot_type='coherence', ax=ax)
ax.set_title('Coherence Map')
plt.tight_layout()
plt.show()

## BT vs Welch Algorithm

`sid.freq_map` supports two algorithms:

- `'bt'` (Blackman-Tukey, default): correlogram-based, configurable lag window
- `'welch'`: Welch averaged periodogram, sub-segment FFT averaging

In [ ]:
result_bt = sid.freq_map(y_tv, u_tv, segment_length=512, algorithm='bt')
result_welch = sid.freq_map(y_tv, u_tv, segment_length=512, algorithm='welch')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sid.map_plot(result_bt, plot_type='magnitude', ax=axes[0])
axes[0].set_title('Blackman-Tukey Algorithm')
sid.map_plot(result_welch, plot_type='magnitude', ax=axes[1])
axes[1].set_title('Welch Algorithm')
plt.tight_layout()
plt.show()

## Segment Length and Overlap Tuning

Shorter segments give better time resolution but worse frequency resolution.
More overlap gives a smoother time axis but requires more computation.

In [ ]:
result_short = sid.freq_map(y_tv, u_tv, segment_length=256, overlap=192)
result_long = sid.freq_map(y_tv, u_tv, segment_length=1024, overlap=768)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sid.map_plot(result_short, plot_type='magnitude', ax=axes[0])
axes[0].set_title('Short Segments (L=256): Good Time Resolution')
sid.map_plot(result_long, plot_type='magnitude', ax=axes[1])
axes[1].set_title('Long Segments (L=1024): Good Freq Resolution')
plt.tight_layout()
plt.show()

## Time-Series Mode: Evolving Output Spectrum

With `u=None`, `sid.freq_map` estimates how the output spectrum changes over time.
Here we simulate a signal whose spectral content drifts.

In [ ]:
rng = np.random.default_rng(30)
N = 4000
a_ts = np.linspace(0.3, 0.9, N)
y_ts = np.zeros(N)
for k in range(1, N):
    y_ts[k] = a_ts[k] * y_ts[k - 1] + rng.standard_normal()

result_ts = sid.freq_map(y_ts, None, segment_length=512)

fig, ax = plt.subplots()
sid.map_plot(result_ts, plot_type='spectrum', ax=ax)
ax.set_title('Time-Series: Output Spectrum Evolves as AR(1) Pole Drifts')
plt.tight_layout()
plt.show()